# Model Training

## Food Price Volatility & Market Risk Prediction

This notebook trains and evaluates machine learning models for classifying near-future food price volatility risk into Low, Medium, and High categories.

The data is split chronologically into training, validation, and final test periods to respect the time-dependent nature of the problem.

In [22]:
import pandas as pd
import numpy as np

print("Model training notebook initialized.")

Model training notebook initialized.


In [23]:
import pandas as pd
import numpy as np

years = [2022, 2023, 2024, 2025, 2026]

dfs = []

for year in years:
    df_year = pd.read_csv(
        f"../data/processed/cleaned_{year}.csv"
    )
    df_year["Arrival_Date"] = pd.to_datetime(df_year["Arrival_Date"])
    dfs.append(df_year)

df = pd.concat(dfs, ignore_index=True)

print("Combined raw modeling source:")
print("Shape:", df.shape)
print("Date range:", df["Arrival_Date"].min(), "to", df["Arrival_Date"].max())

Combined raw modeling source:
Shape: (2522952, 11)
Date range: 2022-01-01 00:00:00 to 2026-04-20 00:00:00


In [30]:
daily_df = (
    df.groupby(
        ["Commodity", "Market", "Arrival_Date"],
        as_index=False
    )
    .agg(
        Daily_Modal_Price=("Modal_Price", "median")
    )
)

print("Daily dataset shape:", daily_df.shape)

print("\nColumns:")
print(daily_df.columns.tolist())

print("\nMissing values:")
print(daily_df.isna().sum())

Daily dataset shape: (2433600, 4)

Columns:
['Commodity', 'Market', 'Arrival_Date', 'Daily_Modal_Price']

Missing values:
Commodity            0
Market               0
Arrival_Date         0
Daily_Modal_Price    0
dtype: int64


In [31]:
daily_df = daily_df.sort_values(
    ["Commodity", "Market", "Arrival_Date"]
).reset_index(drop=True)

for lag in [1, 3, 7]:
    daily_df[f"Lag_{lag}_Price"] = (
        daily_df
        .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
        .shift(lag)
    )

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Lag_1_Price",
            "Lag_3_Price",
            "Lag_7_Price"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        ["Lag_1_Price", "Lag_3_Price", "Lag_7_Price"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Lag_1_Price  Lag_3_Price  \
0      Onion  A lot   2022-05-24              280.0          NaN          NaN   
1      Onion  A lot   2022-05-25              280.0        280.0          NaN   
2      Onion  A lot   2022-05-26              200.0        280.0          NaN   
3      Onion  A lot   2022-05-27              245.0        200.0        280.0   
4      Onion  A lot   2022-05-28              245.0        245.0        280.0   
5      Onion  A lot   2022-05-29              220.0        245.0        200.0   
6      Onion  A lot   2022-05-30              270.0        220.0        245.0   
7      Onion  A lot   2022-05-31              200.0        270.0        245.0   
8      Onion  A lot   2022-06-01              245.0        200.0        220.0   
9      Onion  A lot   2022-06-02              401.0        245.0        270.0   
10     Onion  A lot   2022-06-03              150.0        401.0        200.0   
11     Onion  A lot   2022-0

In [26]:
daily_df["Price_Change_1D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_1_Price"]
) - 1

daily_df["Price_Change_3D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_3_Price"]
) - 1

daily_df["Price_Change_7D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_7_Price"]
) - 1

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Price_Change_1D",
            "Price_Change_3D",
            "Price_Change_7D"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        [
            "Price_Change_1D",
            "Price_Change_3D",
            "Price_Change_7D"
        ]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Price_Change_1D  \
0      Onion  A lot   2022-05-24              280.0              NaN   
1      Onion  A lot   2022-05-25              280.0         0.000000   
2      Onion  A lot   2022-05-26              200.0        -0.285714   
3      Onion  A lot   2022-05-27              245.0         0.225000   
4      Onion  A lot   2022-05-28              245.0         0.000000   
5      Onion  A lot   2022-05-29              220.0        -0.102041   
6      Onion  A lot   2022-05-30              270.0         0.227273   
7      Onion  A lot   2022-05-31              200.0        -0.259259   
8      Onion  A lot   2022-06-01              245.0         0.225000   
9      Onion  A lot   2022-06-02              401.0         0.636735   
10     Onion  A lot   2022-06-03              150.0        -0.625935   
11     Onion  A lot   2022-06-04              300.0         1.000000   
12     Onion  A lot   2022-06-05              300.0         0.00

In [32]:
daily_df["Price_Change_1D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_1_Price"]
) - 1

daily_df["Price_Change_3D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_3_Price"]
) - 1

daily_df["Price_Change_7D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_7_Price"]
) - 1

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Price_Change_1D",
            "Price_Change_3D",
            "Price_Change_7D"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        [
            "Price_Change_1D",
            "Price_Change_3D",
            "Price_Change_7D"
        ]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Price_Change_1D  \
0      Onion  A lot   2022-05-24              280.0              NaN   
1      Onion  A lot   2022-05-25              280.0         0.000000   
2      Onion  A lot   2022-05-26              200.0        -0.285714   
3      Onion  A lot   2022-05-27              245.0         0.225000   
4      Onion  A lot   2022-05-28              245.0         0.000000   
5      Onion  A lot   2022-05-29              220.0        -0.102041   
6      Onion  A lot   2022-05-30              270.0         0.227273   
7      Onion  A lot   2022-05-31              200.0        -0.259259   
8      Onion  A lot   2022-06-01              245.0         0.225000   
9      Onion  A lot   2022-06-02              401.0         0.636735   
10     Onion  A lot   2022-06-03              150.0        -0.625935   
11     Onion  A lot   2022-06-04              300.0         1.000000   
12     Onion  A lot   2022-06-05              300.0         0.00

In [34]:
daily_df["Rolling_Mean_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=7, min_periods=7).mean()
    )
)

daily_df["Rolling_Std_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=7, min_periods=7).std()
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Rolling_Mean_7D",
            "Rolling_Std_7D"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        ["Rolling_Mean_7D", "Rolling_Std_7D"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Rolling_Mean_7D  \
0      Onion  A lot   2022-05-24              280.0              NaN   
1      Onion  A lot   2022-05-25              280.0              NaN   
2      Onion  A lot   2022-05-26              200.0              NaN   
3      Onion  A lot   2022-05-27              245.0              NaN   
4      Onion  A lot   2022-05-28              245.0              NaN   
5      Onion  A lot   2022-05-29              220.0              NaN   
6      Onion  A lot   2022-05-30              270.0              NaN   
7      Onion  A lot   2022-05-31              200.0       248.571429   
8      Onion  A lot   2022-06-01              245.0       237.142857   
9      Onion  A lot   2022-06-02              401.0       232.142857   
10     Onion  A lot   2022-06-03              150.0       260.857143   
11     Onion  A lot   2022-06-04              300.0       247.285714   
12     Onion  A lot   2022-06-05              300.0       255.14

In [35]:
daily_df["Rolling_Mean_14D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=14, min_periods=14).mean()
    )
)

daily_df["Rolling_Std_14D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=14, min_periods=14).std()
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Rolling_Mean_14D",
            "Rolling_Std_14D"
        ]
    ].head(20)
)

print("\nMissing values:")
print(
    daily_df[
        ["Rolling_Mean_14D", "Rolling_Std_14D"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Rolling_Mean_14D  \
0      Onion  A lot   2022-05-24              280.0               NaN   
1      Onion  A lot   2022-05-25              280.0               NaN   
2      Onion  A lot   2022-05-26              200.0               NaN   
3      Onion  A lot   2022-05-27              245.0               NaN   
4      Onion  A lot   2022-05-28              245.0               NaN   
5      Onion  A lot   2022-05-29              220.0               NaN   
6      Onion  A lot   2022-05-30              270.0               NaN   
7      Onion  A lot   2022-05-31              200.0               NaN   
8      Onion  A lot   2022-06-01              245.0               NaN   
9      Onion  A lot   2022-06-02              401.0               NaN   
10     Onion  A lot   2022-06-03              150.0               NaN   
11     Onion  A lot   2022-06-04              300.0               NaN   
12     Onion  A lot   2022-06-05              300.0

In [36]:
daily_df["Month"] = daily_df["Arrival_Date"].dt.month

daily_df["Day_of_Week"] = daily_df["Arrival_Date"].dt.dayofweek

print(
    daily_df[
        ["Arrival_Date", "Month", "Day_of_Week"]
    ].head(15)
)

print("\nMonth values:")
print(sorted(daily_df["Month"].unique()))

print("\nDay_of_Week values:")
print(sorted(daily_df["Day_of_Week"].unique()))

   Arrival_Date  Month  Day_of_Week
0    2022-05-24      5            1
1    2022-05-25      5            2
2    2022-05-26      5            3
3    2022-05-27      5            4
4    2022-05-28      5            5
5    2022-05-29      5            6
6    2022-05-30      5            0
7    2022-05-31      5            1
8    2022-06-01      6            2
9    2022-06-02      6            3
10   2022-06-03      6            4
11   2022-06-04      6            5
12   2022-06-05      6            6
13   2022-06-06      6            0
14   2022-06-07      6            1

Month values:
[np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]

Day_of_Week values:
[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


In [42]:
feature_columns = [
    "Daily_Modal_Price",
    "Lag_1_Price",
    "Lag_3_Price",
    "Lag_7_Price",
    "Price_Change_1D",
    "Price_Change_3D",
    "Price_Change_7D",
    "Rolling_Mean_7D",
    "Rolling_Std_7D",
    "Rolling_Mean_14D",
    "Rolling_Std_14D",
    "Month",
    "Day_of_Week",
    "Commodity"
]

target_column = "Future_Risk_Level"

print("Number of features:", len(feature_columns))
print("\nFeatures:")
print(feature_columns)

print("\nTarget:")
print(target_column)

Number of features: 14

Features:
['Daily_Modal_Price', 'Lag_1_Price', 'Lag_3_Price', 'Lag_7_Price', 'Price_Change_1D', 'Price_Change_3D', 'Price_Change_7D', 'Rolling_Mean_7D', 'Rolling_Std_7D', 'Rolling_Mean_14D', 'Rolling_Std_14D', 'Month', 'Day_of_Week', 'Commodity']

Target:
Future_Risk_Level


In [43]:
print(daily_df.columns.tolist())

['Commodity', 'Market', 'Arrival_Date', 'Daily_Modal_Price', 'Lag_1_Price', 'Lag_3_Price', 'Lag_7_Price', 'Price_Change_1D', 'Price_Change_3D', 'Price_Change_7D', 'Month', 'Day_of_Week', 'Rolling_Mean_7D', 'Rolling_Std_7D', 'Rolling_Mean_14D', 'Rolling_Std_14D']


In [44]:
# Calculate daily returns
daily_df["Price_Return"] = (
    daily_df.groupby(["Commodity", "Market"])["Daily_Modal_Price"].pct_change()
)

# Robustly cap extreme returns
lower_return = daily_df["Price_Return"].quantile(0.01)
upper_return = daily_df["Price_Return"].quantile(0.99)

daily_df["Return_Capped"] = daily_df["Price_Return"].clip(
    lower=lower_return,
    upper=upper_return
)

# Calculate future 7-observation volatility
daily_df["Future_Volatility_7D"] = (
    daily_df.groupby(["Commodity", "Market"])["Return_Capped"]
    .transform(
        lambda x: x.iloc[::-1]
        .rolling(window=7, min_periods=7)
        .std()
        .iloc[::-1]
    )
)

print("Future volatility created.")
print(daily_df["Future_Volatility_7D"].describe())

Future volatility created.
count    2.384538e+06
mean     9.364687e-02
std      8.885700e-02
min      0.000000e+00
25%      2.519763e-02
50%      6.711238e-02
75%      1.357453e-01
max      5.114112e-01
Name: Future_Volatility_7D, dtype: float64


In [45]:
# Define volatility thresholds
p75 = daily_df["Future_Volatility_7D"].quantile(0.75)
p90 = daily_df["Future_Volatility_7D"].quantile(0.90)

# Create future risk classes
daily_df["Future_Risk_Level"] = pd.cut(
    daily_df["Future_Volatility_7D"],
    bins=[-np.inf, p75, p90, np.inf],
    labels=["Low", "Medium", "High"]
)

print("Risk thresholds:")
print("Low / Medium:", p75)
print("Medium / High:", p90)

print("\nRisk distribution:")
print(daily_df["Future_Risk_Level"].value_counts(dropna=False))

print("\nRisk distribution (%):")
print(
    daily_df["Future_Risk_Level"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


Risk thresholds:
Low / Medium: 0.13574531918384464
Medium / High: 0.22785492405866373

Risk distribution:
Future_Risk_Level
Low       1788404
Medium     357680
High       238454
NaN         49062
Name: count, dtype: int64

Risk distribution (%):
Future_Risk_Level
Low       73.49
Medium    14.70
High       9.80
NaN        2.02
Name: proportion, dtype: float64


In [46]:
# Define chronological periods
train_df = daily_df[
    (daily_df["Arrival_Date"] >= "2022-01-01") &
    (daily_df["Arrival_Date"] <= "2024-12-31")
].copy()

val_df = daily_df[
    (daily_df["Arrival_Date"] >= "2025-01-01") &
    (daily_df["Arrival_Date"] <= "2025-12-31")
].copy()

test_df = daily_df[
    (daily_df["Arrival_Date"] >= "2026-01-01") &
    (daily_df["Arrival_Date"] <= "2026-04-20")
].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain dates:")
print(train_df["Arrival_Date"].min(), "to", train_df["Arrival_Date"].max())

print("\nValidation dates:")
print(val_df["Arrival_Date"].min(), "to", val_df["Arrival_Date"].max())

print("\nTest dates:")
print(test_df["Arrival_Date"].min(), "to", test_df["Arrival_Date"].max())

Train: (1780203, 20)
Validation: (597523, 20)
Test: (55874, 20)

Train dates:
2022-01-01 00:00:00 to 2024-12-31 00:00:00

Validation dates:
2025-01-01 00:00:00 to 2025-12-31 00:00:00

Test dates:
2026-01-01 00:00:00 to 2026-04-20 00:00:00


In [47]:
feature_columns = [
    "Daily_Modal_Price",
    "Lag_1_Price",
    "Lag_3_Price",
    "Lag_7_Price",
    "Price_Change_1D",
    "Price_Change_3D",
    "Price_Change_7D",
    "Rolling_Mean_7D",
    "Rolling_Std_7D",
    "Rolling_Mean_14D",
    "Rolling_Std_14D",
    "Month",
    "Day_of_Week",
    "Commodity"
]

target_column = "Future_Risk_Level"

required_columns = feature_columns + [target_column]

train_model_df = train_df[required_columns].dropna().copy()
val_model_df = val_df[required_columns].dropna().copy()
test_model_df = test_df[required_columns].dropna().copy()

print("Model-ready shapes:")
print("Train:", train_model_df.shape)
print("Validation:", val_model_df.shape)
print("Test:", test_model_df.shape)

print("\nTarget distribution:")
print(train_model_df[target_column].value_counts(normalize=True).round(3))

Model-ready shapes:
Train: (1717831, 15)
Validation: (562771, 15)
Test: (23913, 15)

Target distribution:
Future_Risk_Level
Low       0.75
Medium    0.15
High      0.10
Name: proportion, dtype: float64
